# Trisynapse Memory starter

Put files in `data/sample/` (PDF, Markdown, CSV, code, images, zip, or a folder). This notebook writes a local store under `var/starter-notebook-store`.

## Open

In [73]:
from pathlib import Path
import sys

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
SAMPLE_DIR = (ROOT / "data" / "sample").resolve()
STORE_DIR = (ROOT / "var" / "starter-notebook-store").resolve()
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
STORE_DIR.mkdir(parents=True, exist_ok=True)
src = (ROOT / "src").resolve()
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from trisynapse_memory import MemoryEngine, MemoryNamespace, SourceInput

memory = MemoryEngine.from_env(STORE_DIR, namespace=MemoryNamespace(user_id="starter", project_id="sample"))
print(memory.store_path)
print("sample:", sorted(p.name for p in SAMPLE_DIR.iterdir() if not p.name.startswith(".")))

/Users/shanmukh/Desktop/Projects/trisynapse-memory/var/starter-notebook-store
sample: ['Agent_Memory_Survey.md', 'Booking confirmed.pdf', 'icml_2026_filtered_8.csv']


Completion is optional. Text ingest and search work without it. Set a model if you want extraction, images, or generated answers.

In [74]:
config = memory.get_model_configuration()
print("completion", config.completion.provider, config.completion.model)
print("embedding ", config.embedding.provider, config.embedding.model)

# from trisynapse_memory import ProviderSelection
# config.completion = ProviderSelection(provider="anthropic", model="claude-sonnet-4-5")
# memory.set_model_configuration(config)

completion none None
embedding  sentence-transformers all-MiniLM-L6-v2


## Ingest

In [75]:
note = memory.add("Release notes must name the owner of Project Atlas.", episode_id="policy:release")
memory.ingest_messages(
    [
        {"id": "m1", "role": "user", "content": "When did Project Atlas launch?"},
        {"id": "m2", "role": "assistant", "content": "I will check memory after ingest."},
    ],
    episode_id="chat:starter",
)

pending = [
    SourceInput(
        kind="directory" if path.is_dir() else "file",
        path=str(path),
        source_key=f"sample:{path.name}",
        title=path.name,
    )
    for path in sorted(SAMPLE_DIR.iterdir())
    if not path.name.startswith(".") and path.name not in {".DS_Store", "README.md"}
]
if not pending:
    pending = [
        SourceInput(kind="text", text="Project Atlas launched on 14 May 2026.", source_key="inline-atlas", title="Atlas")
    ]

run = memory.ingest_many(pending)
print(run.status)
for result in run.results:
    print(result.status, result.kind, result.source_key, len(result.delta_ids), result.error or "")

completed
skipped file sample:Agent_Memory_Survey.md 26 
skipped file sample:Booking confirmed.pdf 2 
skipped file sample:icml_2026_filtered_8.csv 11 


## Look around

In [76]:
for source in memory.list_sources():
    print(source.kind, source.chunk_count, source.title)

print()
for delta in memory.list(limit=12).items:
    print(delta.seq, delta.kind, (delta.text or "")[:80].replace("\n", " "))

print()
catalog = memory.memory_catalog()
for helper in catalog.helpers:
    print(helper.id, helper.count)

file 11 icml_2026_filtered_8.csv
file 2 Booking confirmed.pdf
file 26 Agent_Memory_Survey.md
text 1 Inline Atlas brief

2 observation user: When did Project Atlas launch, and who owns it?
3 observation assistant: I will look that up in memory after we ingest the brief.
4 observation Project Atlas launched on 14 May 2026. Maya Chen is the owner of Project Atlas.
5 access Query access q_4f68538212ce6943
6 access Feedback for query q_4f68538212ce6943: helpful
7 extraction Release notes must name the owner and the launch date of Project Atlas.
8 retraction Retracted d_001a03e9e82fb6a23635612ca: replaced by the correction above
9 access Query access q_022fbc9d585e0475
10 access Feedback for query q_022fbc9d585e0475: helpful
11 observation # Memory in the Age of AI Agents — Comprehensive Survey Reference  > **Source Su
12 observation *Key literature references:* - Edge et al. (2024). From Local to Global: GraphRA
13 observation - **Benchmarks** (`benchmarks`) - **Constructing Memory** (`cons

## Search and query

In [77]:
QUESTION = "What are different Agent Memory benchmarks?"

found = memory.search(QUESTION, top_k=8, persist=False)
for hit in found.hits:
    print(f"{hit.score:.3f}", hit.kind, hit.text[:90].replace("\n", " "))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7781.64it/s]


0.888 observation # Memory in the Age of AI Agents — Comprehensive Survey Reference  > **Source Survey Paper
0.860 observation ---  ## 4. Comprehensive Taxonomy Reference  Below is the complete detail, definition para
0.860 observation ---  ## 6. Open-Source Frameworks & Middleware  A consolidated directory of the **25** ope
0.701 observation 61557 | When LLMs Develop Languages: Symbolic Communication for Efficient Multi-Agent Reas
0.657 observation ```bibtex @article{zhang2025memory,   title   = {Memory in the Age of AI Agents},   author
0.648 observation 65507 | Representation Unlearning: Forgetting through Information Compression | Antonio Al
0.625 observation 62245 | Beyond Gemini-3-Pro: Revisiting LLM Routing and Aggregation at Scale | Shengji Tan
0.602 observation 65222 | Forget-It-All: Multi-Concept Machine Unlearning via Concept-Aware Neuron Masking |


In [78]:
answer = memory.query(QUESTION)
print(answer.answer)
print("abstain", answer.abstain)
for citation in answer.citations:
    print("-", citation.delta_id, citation.excerpt[:80].replace("\n", " "))

# Memory in the Age of AI Agents — Comprehensive Survey Reference

> **Source Survey Paper**: *Memory in the Age of AI Agents* (Zhang et al., 2025 · [arXiv:2512.13564](https://arxiv.org/abs/2512.13564))
> **Project GitHub**: [ai_agents_memory](https://github.com/shanmukh/ai_agents_memory)

This markdown compiles **all taxonomy definitions, evaluation benchmarks, middleware frameworks, mathematical foundations, comparative matrices, and future frontiers** from the interactive encyclopedia. No details or citation mappings are left out.

---

## 1. Core Paradigm & Foundations

### The Forms–Functions–Dynamics Cognitive Triangle

A comprehensive understanding of agent memory requires looking through a multidimensional lens, breaking down the topic into three interconnected dimensions:

1. **Forms (What Carries Memory?)**: How memory resides within the agentic architecture. Includes Token-level (discrete in context), Parametric (in parameters), and Latent (continuous hidden states).
2. **Fu

## Correct

`correct()` appends a linked correction. It needs a live delta — skip this if you already corrected `note`.

In [79]:
corrected = memory.correct(
    delta_id=note.id,
    text="Release notes must name the owner and the launch date.",
    reason="include launch date",
)
print(corrected.id, corrected.text)

d_001a03ec1645b554c55f30b2e Release notes must name the owner and the launch date.
